### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [1]:
## Import necessary libraries and modules
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model

from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

## Step 1: Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [2]:
## Step 2: Vector Store
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

## Step 3: MMR Retriever
retriever = vectorstore.as_retriever(search_type = "mmr", search_kwargs = {"k":5})
retriever

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000019AC6ABBCB0>, search_type='mmr', search_kwargs={'k': 5})

In [3]:
## Step 4 : LLM and Prompt
import os
from dotenv import load_dotenv
load_dotenv()

# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
# llm = init_chat_model("openai:o4-mini")
# llm

llm = init_chat_model(model = "openai/gpt-oss-120b", model_provider = "groq", temperature = 0.4)
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019ACC1B82F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019ACC000440>, model_name='openai/gpt-oss-120b', temperature=0.4, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
## Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"
Expanded query:
""")

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\nExpanded query:\n')
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019ACC1B82F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019ACC000440>, model_name='openai/gpt-oss-120b', temperature=0.4, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)
| StrOutputParser()

In [5]:
## Invoke the chain with the user query
query_expansion_chain.invoke({"query":"Langchain memory"})

'**Expanded query**\n\n```\nLangChain memory \nOR LangChain memory management \nOR LangChain conversation memory \nOR LangChain session memory \nOR LangChain context storage \nOR LangChain state handling \nOR LangChain memory modules \nOR LangChain memory classes \nOR LangChain memory types \nOR ConversationBufferMemory \nOR ConversationSummaryMemory \nOR ConversationBufferWindowMemory \nOR VectorStoreRetrieverMemory \nOR CombinedMemory \nOR ChatMessageHistory \nOR BaseMemory \nOR LLMChain memory \nOR Agent memory LangChain \nOR persistent memory LangChain \nOR in‑memory cache LangChain \nOR vector store memory LangChain \nOR LangChain memory API \nOR LangChain memory tutorial \nOR LangChain memory example code \nOR LangChain memory documentation \nOR LangChain memory architecture \nOR LangChain memory for chatbots \nOR LangChain memory for agents \nOR LangChain memory for large language models \nOR LangChain memory persistence \nOR LangChain memory caching \nOR LangChain memory retrie

In [6]:
## RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain = create_stuff_documents_chain(llm = llm, prompt = answer_prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {input}\n')
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019ACC1B82F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019ACC000440>, model_name='openai/gpt-oss-120b', temperature=0.4, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)
| StrOutput

In [7]:
## Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | document_chain
)

## Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

**Expanded query**

```json
{
  "input": "What memory mechanisms does LangChain provide? | LangChain memory types | LangChain memory modules | LangChain memory classes | LangChain state management | LangChain conversation memory | LangChain buffer memory | LangChain summary memory | LangChain window memory | LangChain token‑buffer memory | LangChain vector‑store memory | LangChain Redis memory | LangChain SQL memory | LangChain persistent memory | LangChain short‑term memory | LangChain long‑term memory | LangChain BaseMemory subclasses | ConversationBufferMemory | ConversationSummaryMemory | ConversationBufferWindowMemory | ConversationTokenBufferMemory | ConversationSummaryBufferMemory | VectorStoreRetrieverMemory | RedisChatMessageHistory | SQLChatMessageHistory | memory persistence in LangChain | memory serialization in LangChain | LLM chain memory support | LangChain agents memory handling"
}
```
✅ Answer:
 LangChain’s built‑in memory modules include:

- **ConversationBufferMemory

In [8]:
## Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

**Expanded query**

```json
{
  "input": "CrewAI agents OR CrewAI autonomous agents OR AI crew members OR AI‑driven crew OR multi‑agent system OR multi‑agent framework OR autonomous AI agents OR LLM agents OR AI‑powered agents OR AI workflow automation OR AI agent orchestration OR AI agent collaboration OR AI agent hierarchy OR AI agent roles OR AI crew management OR AI crew coordination OR AI team of agents OR AI orchestrator OR AI crew architecture OR CrewAI platform OR CrewAI SDK OR CrewAI API OR CrewAI documentation OR CrewAI use cases OR AI‑driven teamwork OR agentic AI OR AI task delegation OR AI‑enabled crew"
}
```
✅ Answer:
 CrewAI agents are autonomous “specialist” AIs that work together in a structured crew.  
- **Defined role** – each agent is assigned a specific function such as researcher, planner, executor, analyst, etc.  
- **Semi‑independent operation** – an agent carries out its part of the workflow on its own but within the context of the crew’s overall plan.  
- **Co